In [1]:
"""
Parallelized local evaluation — node-level balanced accuracy.
"""
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm
import networkx as nx


In [2]:
# --- Graph utilities (must be top-level for pickling) ---
def graph_nodes_representation(graph, nodelist):
    arr = np.asarray(nx.adjacency_matrix(graph, nodelist=nodelist).todense())
    return tuple(int(x) for x in arr.flatten())


def create_graph_label():
    graph_label = {
        nx.DiGraph([("X", "Y"), ("v", "X"), ("v", "Y")]): "Confounder",
        nx.DiGraph([("X", "Y"), ("X", "v"), ("Y", "v")]): "Collider",
        nx.DiGraph([("X", "Y"), ("X", "v"), ("v", "Y")]): "Mediator",
        nx.DiGraph([("X", "Y"), ("v", "X")]): "Cause of X",
        nx.DiGraph([("X", "Y"), ("v", "Y")]): "Cause of Y",
        nx.DiGraph([("X", "Y"), ("X", "v")]): "Consequence of X",
        nx.DiGraph([("X", "Y"), ("Y", "v")]): "Consequence of Y",
        nx.DiGraph({"X": ["Y"], "v": []}): "Independent",
    }
    nodelist = ["v", "X", "Y"]
    adjacency_label = {
        graph_nodes_representation(g, nodelist): l
        for g, l in graph_label.items()
    }
    return graph_label, adjacency_label


def get_labels(adj_df, adjacency_label):
    result = {}
    for v in adj_df.columns.drop(["X", "Y"]):
        sub = adj_df.loc[[v, "X", "Y"], [v, "X", "Y"]]
        key = tuple(int(x) for x in sub.values.flatten())
        result[v] = adjacency_label.get(key, "Independent")
    return result


def _eval_single(args):
    name, nodes, pred_lookup, true_adj_np, adjacency_label = args
    p = len(nodes)

    # Ground truth: directly from y_test adjacency matrix
    A_true = pd.DataFrame(true_adj_np, columns=nodes, index=nodes)
    true_labels = get_labels(A_true, adjacency_label)

    # Prediction: reconstruct from flat parquet
    adj = np.zeros((p, p), dtype=int)
    for i, ni in enumerate(nodes):
        for j, nj in enumerate(nodes):
            adj[i, j] = int(pred_lookup.get(f"{name}_{ni}_{nj}", 0))
    A_pred = pd.DataFrame(adj, columns=nodes, index=nodes)
    pred_labels = get_labels(A_pred, adjacency_label)

    return list(pred_labels.values()), list(true_labels.values())


def evaluate_predictions(X_test, y_test, y_pred_path, n_workers=None):
    """
    Args:
        X_test: dict {name: DataFrame} — for column names
        y_test: dict {name: DataFrame} — ground truth adjacency matrices
        y_pred_path: path to prediction parquet
    """
    import multiprocessing as mp
    if n_workers is None:
        n_workers = max(1, mp.cpu_count() - 1)

    y_pred_df = pd.read_parquet(y_pred_path)
    pred_lookup = dict(zip(y_pred_df.iloc[:, 0], y_pred_df["prediction"]))

    _, adjacency_label = create_graph_label()

    args = [
        (name, list(X_test[name].columns), pred_lookup,
         y_test[name].values.astype(int), adjacency_label)
        for name in X_test.keys()
    ]

    print(f"Evaluating {len(args)} graphs with {n_workers} workers...")
    all_pred, all_true = [], []

    if n_workers > 1:
        with ProcessPoolExecutor(max_workers=n_workers) as pool:
            for pred_labels, true_labels in tqdm(
                pool.map(_eval_single, args, chunksize=64), total=len(args)
            ):
                all_pred.extend(pred_labels)
                all_true.extend(true_labels)
    else:
        for a in tqdm(args):
            pred_labels, true_labels = _eval_single(a)
            all_pred.extend(pred_labels)
            all_true.extend(true_labels)

    score = balanced_accuracy_score(all_true, all_pred)

    y_pred_s, y_true_s = pd.Series(all_pred), pd.Series(all_true)
    print("\nPer-class accuracy:")
    for label in sorted(y_true_s.unique()):
        mask = y_true_s == label
        acc = np.mean(y_pred_s[mask] == label)
        print(f"  {label:25s}: {acc:.4f}  (n={mask.sum()})")

    print(f"\nBalanced Accuracy: {score:.4f}")
    return score


In [3]:
# --- Usage ---
X_test = pd.read_pickle("data/X_test_reduced.pickle")
y_test = pd.read_pickle("data/y_test_reduced.pickle")

In [4]:
X_train = pd.read_pickle("data/X_train.pickle")
len(X_train)

23500

In [5]:
len(X_train), len(X_test)

(23500, 1880)

In [6]:
score = evaluate_predictions(
    X_test,
    y_test,
    y_pred_path="prediction/prediction_30_b16_lr1e3_n1000_5ch_noobs.parquet",
)

Evaluating 1880 graphs with 7 workers...


  0%|          | 0/1880 [00:00<?, ?it/s]


Per-class accuracy:
  Cause of X               : 0.7681  (n=983)
  Cause of Y               : 0.8154  (n=2145)
  Collider                 : 0.6256  (n=446)
  Confounder               : 0.6818  (n=682)
  Consequence of X         : 0.7467  (n=1571)
  Consequence of Y         : 0.7625  (n=682)
  Independent              : 0.8502  (n=4494)
  Mediator                 : 0.5466  (n=483)

Balanced Accuracy: 0.7246
